<a href="https://colab.research.google.com/github/fernandosuarez89/fernandosuarez89.github.io/blob/main/Ticket_Jumbo_x_Mail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pdfplumber
#link de descarga https://pay.cencosud.com.ar/ords/api/restdbproxy/edoc/v1/vtex/bill/jumboargentinaj5202martinez/v27569338JRSA-01?action=download

In [ ]:
import pdfplumber
import pandas as pd
import re

pdf_path = "factura-v32055515JRSA-01-9495.pdf"

with pdfplumber.open(pdf_path) as pdf:
    text = "\n".join(page.extract_text() for page in pdf.pages if page.extract_text())

lines = [line.strip() for line in text.split('\n') if line.strip()]

# 1. Fecha y número de factura
fecha_factura, numero_factura = None, None
for i, line in enumerate(lines):
    if "IVA RESP. INSCRIPTO" in line:
        for j in range(i+1, i+5):
            match = re.search(r"(\d{2}/\d{2}/\d{4}) \d{2}:\d{2}:\d{2}\s+NRO\.COMP:\s+([\d\-]+)", lines[j])
            if match:
                fecha_factura, numero_factura = match.group(1), match.group(2)
                break
        break

# 2. Extraer productos
start_idx, end_idx = None, None
for i, line in enumerate(lines):
    if re.search(r"Consumidor final", line, re.IGNORECASE):
        start_idx = i + 1
    elif "SUBTOTAL SIN DESCUENTOS" in line:
        end_idx = i
        break

product_lines = lines[start_idx:end_idx] if start_idx and end_idx else []

products = []
i = 0
while i < len(product_lines):
    description = product_lines[i]
    line2 = product_lines[i+1] if i+1 < len(product_lines) else ""
    line3 = product_lines[i+2] if i+2 < len(product_lines) else ""

    qpe_match = re.search(r"([\d.,]+)x([\d.,]+)\s*/\s*(\d+)\s+([\d.,]+)", line2)
    if not qpe_match:
        i += 1
        continue

    quantity = float(qpe_match.group(1).replace(',', '.'))
    unit_price = float(qpe_match.group(2).replace('.', '').replace(',', '.'))
    ean = qpe_match.group(3)
    subtotal = float(qpe_match.group(4).replace('.', '').replace(',', '.'))

    discount = 0.0
    discount_legend = None

    is_discount_line = (
    bool(re.search(r'(Descuento|2do|3x2|OFERTA|2x1|%|SHELL BOX|Promo 15000 Tcom)', line3, re.IGNORECASE)) and
    bool(re.search(r"-[\d.,]+$", line3.strip()))
)

    if is_discount_line:
        # Extraer solo texto, sin monto
        legend_text_match = re.match(r"([^\-]+?)(?:\s+-[\d.,]+)?$", line3.strip())
        discount_legend = legend_text_match.group(1).strip() if legend_text_match else None
        discount_match = re.search(r"-([\d.,]+)", line3)
        if discount_match:
            discount = float(discount_match.group(1).replace('.', '').replace(',', '.'))
        i += 3
    else:
        i += 2

    # 3. Calcular % real de descuento
    discount_pct = 0
    if discount > 0 and subtotal > 0:
        discount_pct = (discount / subtotal)

    # 4. Clasificación
    discount_type = ""
    if discount_legend:
        legend_lower = discount_legend.lower()
        if "2do al 70%" in legend_lower:
            expected_pct = 0.35  # el segundo al 70% → equivale a 35% sobre subtotal de 2
            discount_type = "ok" if abs(discount_pct - expected_pct) < 1.0 else "check discount"
        elif "%" in legend_lower:
            perc_match = re.search(r"(\d+)%", legend_lower)
            if perc_match:
                perc = int(perc_match.group(1))/100
                discount_type = "ok" if abs(discount_pct - perc) < 1.0 else "check discount"
            else:
                discount_type = "percentage unclear"
        elif "3x2" in legend_lower:
            expected_pct = 0.3333333
            discount_type = "ok" if abs(discount_pct - expected_pct) < 2.0 else "check discount"
        elif "2x1" in legend_lower:
            expected_pct = 0.5
            discount_type = "ok" if abs(discount_pct - expected_pct) < 2.0 else "check discount"
        else:
            discount_type = "custom promo"
    elif discount > 0:
        discount_type = "implícito"

    products.append({
        "Description": description,
        "Quantity": quantity,
        "Unit Price": unit_price,
        "EAN": ean,
        "Subtotal": subtotal,
        "Discount": discount,
        "Discount %": discount_pct if discount > 0 else 0,
        "Discount Legend": discount_legend,
        "Subtotal After Discount": subtotal - discount,
        "Discount Type": discount_type
    })

df = pd.DataFrame(products)

# 5. Total facturado
total_facturado = None
for line in lines:
    if re.match(r"^TOTAL\s+[\d.,]+$", line):
        total_str = line.split()[-1]
        total_facturado = float(total_str.replace('.', '').replace(',', '.'))
        break


# df.iloc[21]

In [ ]:
# lines

In [ ]:
df['subtotal_calculado'] = round(df['Quantity'] * df['Unit Price'] * (1-df['Discount %']),2)
df['diff_subtotal'] = df['subtotal_calculado'] - df['Subtotal After Discount']
df['Total Invoiced'] = total_facturado
df['Total_calculated'] = df['subtotal_calculado'].sum()
df['Difference'] = df['Total Invoiced'] - df['Total_calculated']
pd.options.display.float_format = '{:.2f}'.format
# df[(df['diff_subtotal'] > 1) | (df['diff_subtotal'] < -1)]
df

,Description,Quantity,Unit Price,EAN,Subtotal,Discount,Discount %,Discount Legend,Subtotal After Discount,Discount Type,subtotal_calculado,diff_subtotal,Total Invoiced,Total_calculated,Difference
0,"Crema Original Batir LAS TRES NINAS 200 (21,00)",2.00,3350.00,7798338291087,6700.00,2345.00,0.35,PR PRIME 2do al 70% TRES NI,4355.00,ok,4355.00,0.00,124633.90,124633.90,0.00
1,"Leche Entera 3% Casanto 1L SCH 1000 ml. (21,00)",2.00,2200.00,7790742192301,4400.00,830.00,0.19,OFERTA Casanto,3570.00,custom promo,3570.00,0.00,124633.90,124633.90,0.00
2,"toafemSIEMPRELIBRE aplus nyd sua ala 16 (21,00)",1.00,7900.00,7790010002677,7900.00,1580.00,0.20,SHELL BOX,6320.00,custom promo,6320.00,0.00,124633.90,124633.90,0.00
3,"beb. Hierb. TERMA serrano cero 1.75lt B (21,00)",1.00,3800.00,7790950144093,3800.00,0.00,0.00,None,3800.00,,3800.00,0.00,124633.90,124633.90,0.00
4,"Te manzanilla c/sob C&Co 25 saq 50 gr C (21,00)",2.00,1550.00,7792052072939,3100.00,620.00,0.20,SHELL BOX,2480.00,custom promo,2480.00,0.00,124633.90,124633.90,0.00
5,"Harina CASERITA leudante PAQ 1000 gr. (10,50)",1.00,1850.00,7792590001064,1850.00,370.00,0.20,SHELL BOX,1480.00,custom promo,1480.00,0.00,124633.90,124633.90,0.00
6,"soda CUISINE & CO 1.75lt BOT 1750 ml. (21,00)",2.00,2100.00,7792052066013,4200.00,1470.00,0.35,PR PRIME 2do al 70% CUISINE,2730.00,ok,2730.00,0.00,124633.90,124633.90,0.00
7,"agua CUISINE & CO Sin Gas 6.25lt BDN 62 (21,00)",6.00,4550.00,7792052085991,27300.00,9555.00,0.35,PR PRIME 2do al 70% CUISINE,17745.00,ok,17745.00,0.00,124633.90,124633.90,0.00
8,"BEER LANDVIK 4,5 LATA 500CL LAT 500 ml (21,00)",4.00,1700.00,7807910037712,6800.00,1360.00,0.20,SHELL BOX,5440.00,custom promo,5440.00,0.00,124633.90,124633.90,0.00
9,"maquina desechable BIC soleil rosynar 2 (21,00)",1.00,4900.00,70330731769,4900.00,1225.00,0.25,25% BIC,3675.00,ok,3675.00,0.00,124633.90,124633.90,0.00


In [ ]:
# 1. Basic statistics
print("===== Global Summary =====")
print("Number of products:", len(df))
print("Min Subtotal:", df['Subtotal After Discount'].min())
print("Max Subtotal:", df['Subtotal After Discount'].max())
print("Average Subtotal:", df['Subtotal After Discount'].mean())
print()

# 2. Grouped invoice summary
print("===== Invoice Summary =====")
df['Invoice Date'] = fecha_factura
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'], format='%d/%m/%Y')
df['Invoice Number'] = numero_factura
df['Supermarket'] = 'Jumbo'
df['discount rounded'] = round(df['Discount %'],2)

# Remove possible duplicates by selecting first row per invoice
invoice_summary = df.drop_duplicates(subset=['Invoice Date', 'Invoice Number'])[
    ['Supermarket','Invoice Date', 'Invoice Number', 'Total Invoiced', 'Total_calculated','Difference']
].copy()
print(invoice_summary.to_string(index=False))

print()
print("===== Discounts Applied =====")
print(df.drop_duplicates(subset=['Discount Type','Discount Legend','discount rounded'])[['Discount Type','Discount Legend','discount rounded']].to_string(index=False))
#

===== Global Summary =====
Number of products: 24
Min Subtotal: 891.11
Max Subtotal: 17745.0
Average Subtotal: 5193.079166666666

===== Invoice Summary =====
Supermarket Invoice Date Invoice Number  Total Invoiced  Total_calculated  Difference
      Jumbo   2026-10-29 10540-00276118       124633.90         124633.90        0.00

===== Discounts Applied =====
Discount Type             Discount Legend  discount rounded
           ok PR PRIME 2do al 70% TRES NI              0.35
 custom promo              OFERTA Casanto              0.19
 custom promo                   SHELL BOX              0.20
                                     None              0.00
           ok PR PRIME 2do al 70% CUISINE              0.35
           ok                     25% BIC              0.25
           ok      PR PRIME 25% VEGETALES              0.25


In [ ]:
# Ask for confirmation
confirm = input("Do you want to upload the CSV to Google Drive? (y/n): ").strip().lower()

#reorder columns
df = df[[
    'Description','Quantity','Unit Price','EAN','Subtotal','Discount %','Subtotal After Discount','subtotal_calculado','Invoice Date','Invoice Number','Total Invoiced','Total_calculated','Difference','Supermarket',
]]

if confirm == 'y':
    from google.colab import drive
    drive.mount('/content/drive')

    import os

    output_dir = "/content/drive/My Drive/Invoices"
    os.makedirs(output_dir, exist_ok=True)  # ensure folder exists

    output_filename = f"{df['Supermarket'].iloc[0]}_{df['Invoice Number'].iloc[0]}_{df['Invoice Date'].iloc[0].strftime('%Y%m%d')}.csv"
    output_path = os.path.join(output_dir, output_filename)

    df.to_csv(output_path, index=False)
    print("Saved as:", output_filename)
else:
    print("Upload cancelled.")


Do you want to upload the CSV to Google Drive? (y/n): y
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved as: Jumbo_10540-00276118_20261029.csv
